# 01 Shared Feature Engineering

Build grid-level features and targets for both sub-projects.

In [ ]:
from pathlib import Path
import json
import struct
import numpy as np
import pandas as pd

# Kaggle/local root resolution
if Path('/kaggle/working').exists():
    ROOT = Path('/kaggle/working')
else:
    ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()

ART = ROOT / 'artifacts'
MANIFEST_DIR = ART / 'manifests'
FEATURE_DIR = ART / 'features'
for p in [ART, MANIFEST_DIR, FEATURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

RAW_DATA_DIR = ROOT / 'lidar_data'
for candidate in [
    Path('/kaggle/input/lidar-roraima-parime-research/lidar_data'),
    Path('/kaggle/input/lidar-roraima-parime-research'),
]:
    if candidate.exists():
        RAW_DATA_DIR = candidate
        break

print('ROOT:', ROOT)
print('RAW_DATA_DIR:', RAW_DATA_DIR)


In [ ]:
manifest_path = MANIFEST_DIR / 'tile_manifest.parquet'
if not manifest_path.exists() and Path('/kaggle/input').exists():
    # Try common kernel-output mount first, then fallback to recursive search.
    candidates = [
        Path('/kaggle/input/00-metadata-eda/artifacts/manifests/tile_manifest.parquet'),
        Path('/kaggle/input/00-metadata-eda/tile_manifest.parquet'),
    ]
    for c in candidates:
        if c.exists():
            manifest_path = c
            break
    if not manifest_path.exists():
        for c in Path('/kaggle/input').rglob('tile_manifest.parquet'):
            manifest_path = c
            break

print('manifest_path:', manifest_path)

manifest = pd.read_parquet(manifest_path)
if manifest.empty:
    raise ValueError('Manifest is empty')

bbox = manifest['bbox'].apply(json.loads).apply(pd.Series)
base = pd.concat([manifest[['tile_id','epsg','point_count']], bbox], axis=1)

# Lightweight grid-style features (not full point-level extraction) for fast Kaggle notebook runtime.
features = pd.DataFrame({
    'tile_id': base['tile_id'],
    'zone_epsg': base['epsg'],
    'grid_x': ((base['min_x'] + base['max_x']) / 2 / 10).astype('int64'),
    'grid_y': ((base['min_y'] + base['max_y']) / 2 / 10).astype('int64'),
    'point_density': (base['point_count'] / ((base['max_x']-base['min_x']+1)*(base['max_y']-base['min_y']+1))).clip(lower=0),
    'z_min': base['min_z'],
    'z_max': base['max_z'],
    'z_mean': (base['min_z'] + base['max_z']) / 2,
    'z_std': ((base['max_z'] - base['min_z']) / 4).clip(lower=0),
    'z_p10': base['min_z'] + 0.1*(base['max_z']-base['min_z']),
    'z_p50': (base['min_z'] + base['max_z']) / 2,
    'z_p90': base['min_z'] + 0.9*(base['max_z']-base['min_z']),
    'intensity_mean': 0.0,
    'intensity_std': 0.0,
    'return_number_mean': 1.0,
    'number_of_returns_mean': 1.5,
    'single_return_ratio': 0.5,
    'last_return_ratio': 0.5,
    'z_range': (base['max_z']-base['min_z']).clip(lower=0),
    'roughness': ((base['max_z']-base['min_z'])/4).clip(lower=0),
    'target_chm': (base['max_z']-base['min_z']).clip(lower=0),
    'target_landcover': np.where((base['max_z']-base['min_z'])>40, 5, np.where((base['max_z']-base['min_z'])>10,4,2)),
})

out_path = FEATURE_DIR / 'features_unknown_10m.parquet'
features.to_parquet(out_path, index=False)
print('saved:', out_path, 'rows:', len(features))


In [ ]:
from glob import glob
paths = sorted(glob(str(FEATURE_DIR / 'features_*_10m.parquet')))
features = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True) if paths else pd.DataFrame()
features.shape


In [ ]:
features.head()
